In [2]:
# Imports

import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

# Connect to OpenAI

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key found - please check your .env file")

openai = OpenAI()

# Step 1: Set up the folders

LOGS_DIR = Path("logs")        # drop new .log / .txt files here
TICKETS_DIR = Path("tickets")  # generated tickets go here
LOGS_DIR.mkdir(exist_ok=True)
TICKETS_DIR.mkdir(exist_ok=True)

# Step 2: Create your prompts

system_prompt = """
You are a senior QA engineer who turns raw test failure logs into clear bug tickets.

Respond in markdown with exactly these sections:
- **Title**: one line, specific enough to find in a bug tracker
- **Summary**: no more than two sentences
- **Severity**: Critical, High, Medium or Low, with a one-line reason
- **Steps to Reproduce**: a numbered list
- **Expected Result**
- **Actual Result**
- **Likely Component**

Use only the information in the log. Do not invent details.
If information for a section is missing, write "Not provided".
Do not wrap the markdown in a code block.
"""

def user_prompt_for(log_text):
    return "Create a bug ticket from this test log.\n\n" + log_text

# Step 3: Call OpenAI for one log

def create_ticket(log_text):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(log_text)},
    ]
    response = openai.chat.completions.create(
        model="gpt-6-luna",
        reasoning_effort="medium",
        messages=messages,
    )
    return response.choices[0].message.content

# Step 4: Create a ticket for every log that doesn't have one yet
# (to regenerate a ticket, delete it from the tickets folder and run again)

new_tickets = 0

for log_file in sorted(LOGS_DIR.iterdir()):
    if log_file.suffix not in (".log", ".txt"):
        continue
    ticket_file = TICKETS_DIR / f"{log_file.stem}.md"
    if ticket_file.exists():
        continue
    ticket = create_ticket(log_file.read_text(encoding="utf-8"))
    ticket_file.write_text(ticket, encoding="utf-8")
    new_tickets += 1
    display(Markdown(f"### {log_file.name} → {ticket_file}\n\n{ticket}"))

if new_tickets == 0:
    print("No new logs - every log already has a ticket.")

### checkout_saved_card.log → tickets\checkout_saved_card.md

**Title**: Saved-card checkout remains PENDING after payment authorization times out

**Summary**: In staging on build 3.14.0-rc2, `test_checkout_with_saved_card` failed on all three retries after the payment authorization request timed out. The order remained PENDING instead of becoming CONFIRMED; the test passed on build 3.13.2.

**Severity**: High — Saved-card checkout did not reach the expected confirmed order status in the test.

**Steps to Reproduce**:
1. Log in as `test.buyer01@example.com`.
2. Add SKU-10482 to the basket with quantity 2.
3. Select the saved card ending in 4242.
4. Submit the payment authorization request.

**Expected Result**: Order status is CONFIRMED.

**Actual Result**: The authorization request returned a 504 Gateway Timeout after 30,002 ms; the order status was PENDING.

**Likely Component**: payment-service

### guest_checkout_payment.log → tickets\guest_checkout_payment.md

**Title**: Guest card checkout remains PENDING after payment authorization times out

**Summary**: In staging build 3.14.0-rc2 on Safari 19, guest checkout card authorization returned a 504 after 30,001 ms. The confirmation page showed “We're processing your order,” and the test found the order status was PENDING instead of CONFIRMED on both retries.

**Severity**: High — Guest card checkout did not reach the expected confirmed order status.

**Steps to Reproduce**:
1. Start guest checkout.
2. Add SKU-31007 to the basket with quantity 1.
3. Enter a new card ending in 1111.
4. Submit payment authorization.

**Expected Result**: Order status is CONFIRMED.

**Actual Result**: Payment authorization returns 504 Gateway Timeout after 30,001 ms; the confirmation page says “We're processing your order,” and the order status is PENDING.

**Likely Component**: Payment service or its upstream gateway request.

In [3]:
# Step 5: A Gradio UI - pick, upload or paste a log, watch the ticket appear, then save it

import re
import gradio as gr

LOG_SUFFIXES = (".log", ".txt")


def read_log(log_path):
    """Load a log into the text box and suggest a ticket name from the file name."""
    if not log_path:
        return "", ""
    log_file = Path(log_path)
    return log_file.read_text(encoding="utf-8"), log_file.stem


def stream_ticket(log_text):
    """Same as create_ticket, but streams the ticket back as it is written."""
    if not log_text.strip():
        yield "Pick, upload or paste a log first."
        return
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(log_text)},
    ]
    stream = openai.chat.completions.create(
        model="gpt-6-luna",
        reasoning_effort="medium",
        messages=messages,
        stream=True,
    )
    ticket = ""
    for chunk in stream:
        if chunk.choices:
            ticket += chunk.choices[0].delta.content or ""
            yield ticket


def save_ticket(ticket, ticket_name):
    """Save the ticket to the tickets folder, without overwriting an existing one."""
    if not ticket.strip() or "**Title**" not in ticket:
        return "Nothing to save yet - create a ticket first."
    safe_name = re.sub(r"[^\w-]+", "_", ticket_name.strip()).strip("_")
    if not safe_name:
        return "Give the ticket a name first."
    ticket_file = TICKETS_DIR / f"{safe_name}.md"
    if ticket_file.exists():
        return f"**{ticket_file}** already exists - choose another name."
    ticket_file.write_text(ticket, encoding="utf-8")
    return f"Saved to **{ticket_file}**"


sample_logs = [
    (log_file.name, str(log_file))
    for log_file in sorted(LOGS_DIR.iterdir())
    if log_file.suffix in LOG_SUFFIXES
]

with gr.Blocks(title="Bug Ticket Generator") as ui:
    gr.Markdown("# Bug Ticket Generator\nTurn a raw test failure log into a clear bug ticket.")
    with gr.Row():
        with gr.Column():
            log_picker = gr.Dropdown(sample_logs, value=None, label="Pick a log from the logs folder")
            log_upload = gr.File(label="Or upload a log", file_types=list(LOG_SUFFIXES), type="filepath", height=120)
            log_input = gr.Textbox(label="Log", info="Or paste the log here", lines=14, max_lines=14)
            create_button = gr.Button("Create ticket", variant="primary")
        with gr.Column():
            ticket_output = gr.Markdown("_The ticket will appear here._", container=True, min_height=420)
            ticket_name = gr.Textbox(label="Ticket name", info="Saved in the tickets folder as a .md file")
            save_button = gr.Button("Save ticket")
            save_status = gr.Markdown()

    log_picker.change(read_log, inputs=log_picker, outputs=[log_input, ticket_name])
    log_upload.upload(read_log, inputs=log_upload, outputs=[log_input, ticket_name])
    create_button.click(stream_ticket, inputs=log_input, outputs=ticket_output)
    save_button.click(save_ticket, inputs=[ticket_output, ticket_name], outputs=save_status)

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.
